In [1]:
import numpy as np
import casadi as ca
import pandas as pd
import matplotlib.pyplot as plt
import pypesto 
import scipy
# 
from src.kinetics import *
import time
from src.param_estimation import *
import scipy.stats as stats
import seaborn as sns

/Users/gabbi/miniconda3/envs/SaaLab/lib/python3.12/site-packages/pint/compat.py:231: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.26.0)
  import scipy  # noqa: F401


In [2]:
# data_import
balanced_met_df   = pd.read_csv('Data/balanced_metabolites.csv', index_col=0)
imbalanced_met_df = pd.read_csv('Data/imbalanced_metabolites.csv', index_col=0)
prot_df           = pd.read_csv('Data/important_proteins.csv', index_col=0)
flux_df           = pd.read_csv('Data/important_fluxes.csv', index_col=0)
cell_needs_df     = pd.read_csv('Data/cellular_needs.csv', index_col=0)

max_flux  = flux_df.max(axis = 1)
max_balanced = balanced_met_df.max(axis = 1)
max_measures = pd.concat([max_flux, max_balanced]).to_dict()

# bounds
imbalanced_bounds = {}
total_min = 1e12
for met in imbalanced_met_df.index:
    
    imbalanced_bounds[met] = (imbalanced_met_df.loc[met].min()* 0.8, imbalanced_met_df.loc[met].max()    * 1.2)
    if imbalanced_bounds[met][0] < total_min:
        total_min = imbalanced_bounds[met][0]
imbalanced_bounds["C_pi"] = (total_min, 10.0)  # Pi is often in the mM range, but can vary widely; set a broad range.



conditions_enzymes = [prot_df.iloc[:, i].to_dict() for i in range(prot_df.shape[1])]
obs_df             = pd.concat([balanced_met_df, flux_df], axis=0)  # axis=0, not 1
conditions_obs     = [obs_df.iloc[:, i] 
                      for i in range(obs_df.shape[1])]
condition_cell_needs = [cell_needs_df.iloc[:, i].to_dict() for i in range(cell_needs_df.shape[1])]


# Tighter physiological bounds — replaces (1e-12, 100)

max_max_balanced = balanced_met_df.max().max() * 1.2
min_min_balanced = 1e-6
balanced_bounds = {
    "C_g6p": (min_min_balanced, max_max_balanced), 
    "C_f6p": (min_min_balanced, max_max_balanced), 
    "C_fbp": (min_min_balanced, max_max_balanced), 
    "C_dhap": (min_min_balanced, max_max_balanced), 
    "C_g3p": (min_min_balanced, max_max_balanced), 
    "C_pgp": (min_min_balanced, max_max_balanced), 
    "C_3pg": (min_min_balanced, max_max_balanced), 
    "C_2pg": (min_min_balanced, max_max_balanced), 
    "C_pep": (min_min_balanced, max_max_balanced), 
}

model = EcoliCarbonKinetics(
         bounds_imbalanced_mets = imbalanced_bounds,
         bounds_balanced_mets = balanced_bounds,
        )


constants = {
    # v1: PTS (Sistema de fosfotransferasa)
    "kI_pyr_1": 0.5,  # Inhibición por piruvato
    "kA_pep_1": 0.3,  # Afinidad/Activación por PEP
    "v_max_1": 25.739,  # Flujo máximo típico en fase exponencial
    "Ka1_1": 1.0,  # Parámetros de la ecuación compleja del PTS
    "Ka2_1": 0.01,
    "Ka3_1": 1.0,
    "n_g6p_1": 4,  # Cooperatividad/Regulación por G6P
    "K_g6p_1": 0.5,
    # v2: PGI (Glucosa-6-fosfato isomerasa)
    "kI_pep_2": 0.12,  # Fuerte inhibición por PEP
    "Km_g6p_2": 0.48,
    "Km_f6p_2": 0.19,
    "kcat_f_2": 1475.0,  # Reacción muy rápida
    "kcat_r_2": 1000.0,
    # v3: PFK (Fosfofructoquinasa) - Enzima regulatoria clave
    "kI_f6p_3": 0.9,
    "kI_fbp_3": 2.0,
    "kI_gtp_3": 0.8,
    "kI_pep_3": 0.5,  # Inhibidor alostérico principal
    "kI_pi_3": 1.5,
    "kA_adp_3": 0.12,  # Activador alostérico
    "kA_gdp_3": 0.15,
    "Km_f6p_3": 0.16,
    "Km_atp_3": 0.12,
    "Km_fbp_3": 0.5,
    "Km_adp_3": 0.2,
    "kcat_f_3": 580.0,
    "kcat_r_3": 100.0,
    # v4: FBA (Fructosa-bisfosfato aldolasa)
    "kI_3pg_4": 2.0,
    "kI_dhap_4": 0.08,
    "kI_g3p_4": 0.1,
    "kA_pep_4": 1.5,
    "kcat_f_4": 95.0,
    "kcat_r_4": 150.0,
    "Km_fbp_4": 0.3,
    "Km_g3p_4": 0.4,
    "Km_dhap_4": 2.0,
    # v5: TPI (Triosafosfato isomerasa)
    "kcat_f_5": 4300.0,  # Cercana a la perfección catalítica
    "kcat_r_5": 2400.0,
    "Km_dhap_5": 0.61,
    "Km_g3p_5": 1.2,
    # v6: GAPDH (Gliceraldehído-3-fosfato deshidrogenasa)
    "kI_adp_6": 0.8,
    "kI_amp_6": 1.0,
    "kI_atp_6": 0.2,
    "kcat_f_6": 118.0,
    "kcat_r_6": 10.0,
    "Km_g3p_6": 0.21,
    "Km_pi_6": 0.29,
    "Km_nad_6": 0.09,
    "Km_pgp_6": 0.01,
    "Km_nadh_6": 0.06,
    # v7: PGK (Fosfoglicerato quinasa)
    "kA_3pg_7": 0.5,
    "kA_atp_7": 0.3,
    "kcat_f_7": 1150.0,
    "kcat_r_7": 40.0,
    "Km_pgp_7": 0.05,
    "Km_adp_7": 0.1,
    "Km_3pg_7": 0.53,
    "Km_atp_7": 0.3,
    # v8: GPM (Fosfoglicerato mutasa)
    "kI_pi_8": 10.0,  # Inhibición débil por Pi
    "kcat_f_8": 540.0,
    "kcat_r_8": 120.0,
    "Km_3pg_8": 0.2,
    "Km_2pg_8": 1.4,
    # v9: ENO (Enolasa)
    "kcat_f_9": 550.0,
    "kcat_r_9": 210.0,
    "Km_2pg_9": 0.1,
    "Km_pep_9": 0.5,
}


In [ ]:
PARAM_BOUNDS_BY_PREFIX = {
    "kI"   : (1e-3, 2000.0),    # inhibition constants, mM
    "kA"   : (1e-3, 2000.0),    # activation constants, mM
    "Km"   : (1e-3, 2000.0),    # Michaelis constants, mM
    "kcat" : (0.1,  5000.0),  # catalytic rates, s⁻¹
    "v_max": (1.0,  1000.0),   # max velocity, mmol/gDW/h
    "Ka"   : (1e-3, 1000.0),    # PTS affinity constants
    "K_"   : (1e-3, 1000.0),    # other K-type constants
}

def _bounds_for_param(name: str):
    for prefix, bounds in PARAM_BOUNDS_BY_PREFIX.items():
        if name.startswith(prefix):
            return bounds
    return (1e-3, 1000.0)   # fallback

def build_bounds(param_keys):
    lb = np.array([_bounds_for_param(k)[0] for k in param_keys])
    ub = np.array([_bounds_for_param(k)[1] for k in param_keys])
    return lb, ub

lb, ub = build_bounds(model.params_keys)
PARAM_BOUNDS = {model.params_keys[i]: (float(lb[i]), float(ub[i])) 
                for i in range(len(model.params_keys))}

p_opt, best_obj = param_estimation_mealpy(
    model,
    conditions_enzymes = conditions_enzymes,
    conditions_obs     = conditions_obs,
    conditions_cell_needs = condition_cell_needs,
    max_measure = max_measures,
    bounds_params      = PARAM_BOUNDS,
    algorithm          = 'PSO',
    epoch              = 300,
    pop_size           = 40,
)

2026/05/27 12:23:04 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: OriginalPSO(epoch=300, pop_size=40, c1=2.05, c2=2.05, w=0.4)


[INFO] Fitting on 15 variables: {'C_3pg', 'C_pep', 'v_pts', 'v_pgk', 'C_dhap', 'v_eno', 'v_gapA', 'C_f6p', 'v_gpmA', 'v_pgi', 'C_fbp', 'v_pfkB', 'v_fbaA', 'v_tpiA', 'C_g6p'}
Running PSO  (epoch=300, pop_size=40, total evals ≈ 12,000)...


2026/05/27 12:35:06 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 1, Current best: 43.056755717549805, Global best: 43.056755717549805, Runtime: 333.88477 seconds
2026/05/27 12:42:37 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 2, Current best: 43.056755717549805, Global best: 43.056755717549805, Runtime: 451.59013 seconds
2026/05/27 12:48:39 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 3, Current best: 43.056755717549805, Global best: 43.056755717549805, Runtime: 361.55982 seconds
2026/05/27 12:54:51 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 4, Current best: 43.056755717549805, Global best: 43.056755717549805, Runtime: 372.35598 seconds
2026/05/27 01:00:31 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 5, Current best: 43.056755717549805, Global best: 43.056755717549805, Runtime: 340.18607 seconds
2026/05/27 01:08:06 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Proble